In [106]:
import pandas as pd

In [107]:
df = pd.read_parquet("/content/Films.parquet")

In [108]:
df

,title,year,genres
movie_id,,,
1,Toy Story,1995,"[Animation, Children's, Comedy]"
2,Jumanji,1995,"[Adventure, Children's, Fantasy]"
3,Grumpier Old Men,1995,"[Comedy, Romance]"
4,Waiting to Exhale,1995,"[Comedy, Drama]"
5,Father of the Bride Part II,1995,[Comedy]
...,...,...,...
3948,Meet the Parents,2000,[Comedy]
3949,Requiem for a Dream,2000,[Drama]
3950,Tigerland,2000,[Drama]


In [109]:
df["year"] = pd.to_datetime(df["year"], errors='coerce')


mask = df["year"].isna()

df.loc[mask, "year"] = pd.to_datetime("1995")

In [110]:
df

,title,year,genres
movie_id,,,
1,Toy Story,1995-01-01,"[Animation, Children's, Comedy]"
2,Jumanji,1995-01-01,"[Adventure, Children's, Fantasy]"
3,Grumpier Old Men,1995-01-01,"[Comedy, Romance]"
4,Waiting to Exhale,1995-01-01,"[Comedy, Drama]"
5,Father of the Bride Part II,1995-01-01,[Comedy]
...,...,...,...
3948,Meet the Parents,2000-01-01,[Comedy]
3949,Requiem for a Dream,2000-01-01,[Drama]
3950,Tigerland,2000-01-01,[Drama]


In [111]:
mask = df["title"].isna() #
df.loc[mask, "title"] = "Phantom of the Opera, The"

In [112]:
mask = (df["genres"].str[0].str.contains("-")) & ~(df["genres"].str[0].str.contains("Sci-Fi")) & ~(df["genres"].str[0].str.contains("Film-Noir"))

pd.set_option('display.max_rows', 500)
df.loc[mask]

,title,year,genres
movie_id,,,
3785,Scary Movie,2000-01-01,[Comedy--Horror]


In [113]:
df.at[df.loc[mask].index[0], "genres"] = ["Comedy", "Horror"]

In [114]:
df[mask]

,title,year,genres
movie_id,,,
3785,Scary Movie,2000-01-01,"[Comedy, Horror]"


In [115]:
!pip install thefuzz

In [116]:
from thefuzz import fuzz
mask = df["genres"].str[0].apply(lambda x: (fuzz.ratio(x, "Drama") >= 50 and x != "Drama"))
df.loc[mask]

,title,year,genres
movie_id,,,
527,Schindler's List,1993-01-01,[Dramatic]
608,Fargo,1996-01-01,[Dramma]
3317,Wonder Boys,2000-01-01,[Dramma]
3409,Final Destination,2000-01-01,[Dramatic]
3578,Gladiator,2000-01-01,[Dramatic]


In [117]:
valori_drama = [np.array(["Drama"]) for _ in range(5)]

df.loc[mask, "genres"] = valori_drama

In [118]:
#alternative_df = df.copy()

In [119]:
#alternative_df["genres"] = alternative_df["genres"].apply(lambda x: "|".join(x))
#alternative_df

In [120]:
import numpy as np

In [121]:
df["genres"] = df["genres"].apply(lambda x: x if isinstance(x, np.ndarray) else np.array(x))

In [122]:
print(df["genres"].apply(type).value_counts())

genres
<class 'numpy.ndarray'>    3878
<class 'str'>                 5
Name: count, dtype: int64


In [123]:
df['genres'] = df['genres'].apply(lambda x: x.tolist() if hasattr(x, 'tolist') else x)

In [124]:
def normalizza_in_lista(val):
    # Se è NaN o None -> Lista vuota
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return []

    # Se è un numpy array -> Converti in lista Python
    if isinstance(val, np.ndarray):
        return val.tolist()

    # Se è già una lista -> Lasciala così
    if isinstance(val, list):
        return val

    # Se è una stringa singola (errore comune) -> Mettila in una lista
    if isinstance(val, str):
        return [val]

    # Fallback per altri casi
    return [val]

# Applica la correzione
df["genres"] = df["genres"].apply(normalizza_in_lista)

In [125]:
print(df["genres"].apply(type).value_counts())

genres
<class 'list'>    3883
Name: count, dtype: int64


In [126]:
df.to_parquet("Movies_Cleaned_frfr.parquet")
#alternative_df.to_parquet("Film_Cleaned_GenresUnified.parquet")